MLOps — Machine Learning Operations MLOps is the practice of combining Machine Learning + Software Engineering + DevOps to take an ML model from development to deployment, monitoring, and continuous improvement.

In [1]:
!pip install -q mlflow scikit-learn pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 523.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.6/144.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

import mlflow
import mlflow.sklearn

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
data = {
    "message": [
        "Congratulations! You won a free lottery",
        "Win a free iPhone now",
        "Claim your cash prize immediately",
        "You have won 10000 dollars",
        "Click this link to receive your reward",
        "URGENT! You have won a prize",
        "Free recharge available claim now",
        "Congratulations you are the lucky winner",
        "Get your free gift today",
        "You won a shopping voucher",

        "Can you call me when you reach home",
        "Meeting is scheduled for tomorrow",
        "Please send me the project report",
        "What time is the class today",
        "Don't forget to bring your laptop",
        "I will reach college by 10 AM",
        "Can you share the notes",
        "Let's meet after lunch",
        "The training starts at 9 AM",
        "Please submit the assignment today"
        ],

    "label": [
        "spam","spam","spam","spam","spam",
        "spam","spam","spam","spam","spam",
        "ham","ham","ham","ham","ham",
        "ham","ham","ham","ham","ham"
        ]
    }

df = pd.DataFrame(data)

In [4]:
df.head()

,message,label
0,Congratulations! You won a free lottery,spam
1,Win a free iPhone now,spam
2,Claim your cash prize immediately,spam
3,You have won 10000 dollars,spam
4,Click this link to receive your reward,spam


In [5]:
print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["label"].value_counts())

Dataset shape: (20, 2)

Class distribution:
label
spam    10
ham     10
Name: count, dtype: int64


In [6]:
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y )

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 16
Testing samples: 4


In [7]:
model = Pipeline([ ("tfidf", TfidfVectorizer()), ("classifier", LogisticRegression()) ])

model.fit(X_train, y_train)
print("Model training completed!")

Model training completed!


In [8]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00         2
        spam       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



In [9]:
messages = [ "Congratulations you won a free prize", "Please send me today's class notes", "URGENT! Claim your reward now", "Can we meet tomorrow?" ]

predictions = model.predict(messages)

for message, prediction in zip(messages, predictions):
  print(f"Message: {message}")
  print(f"Prediction: {prediction.upper()}")
  print("-" * 50)

Message: Congratulations you won a free prize
Prediction: SPAM
--------------------------------------------------
Message: Please send me today's class notes
Prediction: HAM
--------------------------------------------------
Message: URGENT! Claim your reward now
Prediction: SPAM
--------------------------------------------------
Message: Can we meet tomorrow?
Prediction: HAM
--------------------------------------------------


In [10]:
mlflow.set_experiment("Spam_Message_Detection")

with mlflow.start_run():
  mlflow.log_param("model", "Logistic Regression")
  mlflow.log_param("feature_extraction", "TF-IDF")
  mlflow.log_metric("accuracy", accuracy)
  mlflow.sklearn.log_model( model, "spam_detection_model" )
print("Experiment logged successfully!")

2026/09/21 05:27:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/21 05:27:22 INFO mlflow.store.db.utils: Updating database tables
2026/09/21 05:27:24 INFO mlflow.tracking.fluent: Experiment with name 'Spam_Message_Detection' does not exist. Creating a new experiment.
2026/09/21 05:27:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Experiment logged successfully!


In [11]:
import joblib

joblib.dump(model, "spam_detection_model.pkl")
print("Model saved successfully!")

Model saved successfully!


In [12]:
loaded_model = joblib.load("spam_detection_model.pkl")
print("Model loaded successfully!")

Model loaded successfully!


In [13]:
test_messages = [ "Congratulations! You won a free iPhone", "Please send me the project report", "Claim your cash prize now", "What time is our class tomorrow?" ]

predictions = loaded_model.predict(test_messages)

for message, prediction in zip(test_messages, predictions):
  print("Message:", message)
  print("Prediction:", prediction.upper())
  print("-" * 60)

Message: Congratulations! You won a free iPhone
Prediction: SPAM
------------------------------------------------------------
Message: Please send me the project report
Prediction: HAM
------------------------------------------------------------
Message: Claim your cash prize now
Prediction: SPAM
------------------------------------------------------------
Message: What time is our class tomorrow?
Prediction: HAM
------------------------------------------------------------


In [14]:
!pip install -q fastapi uvicorn

In [15]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

# Create FastAPI application

app = FastAPI(title="Spam Message Detector API")

# Load trained model
model = joblib.load("spam_detection_model.pkl")

# Input format
class MessageRequest(BaseModel):
  message: str

# Home endpoint
@app.get("/")
def home():
  return { "message": "Spam Message Detector API is running" }

# Prediction endpoint
@app.post("/predict")
def predict(request: MessageRequest):
  prediction = model.predict([request.message])[0]
  return { "message": request.message, "prediction": prediction }

In [16]:
!pip install -q nest_asyncio pyngrok

In [17]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
  uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api)
thread.start()
print("FastAPI server started!")

FastAPI server started!


In [18]:
from pyngrok import ngrok

ngrok.set_auth_token("3JcjQvqMTteiDOOdbC8UbpD88qM_6BrWqrFyFuQQXDkrBHp4q")
public_url = ngrok.connect(8000)
print("FastAPI Public URL:")
print(public_url)

FastAPI Public URL:
NgrokTunnel: "https://mosaic-distaste-hatching.ngrok-free.dev" -> "http://localhost:8000"


In [19]:
model_v2 = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2)    )),
         ("classifier", LogisticRegression(
             C=0.5,
             max_iter=1000
             ))
         ])

model_v2.fit(X_train, y_train)
y_pred_v2 = model_v2.predict(X_test)
accuracy_v2 = accuracy_score(y_test, y_pred_v2)
print("Model V2 Accuracy:", accuracy_v2)

Model V2 Accuracy: 1.0


In [20]:
with mlflow.start_run(run_name="Spam_Model_V2_Registry") as run:
  # Log parameters
  mlflow.log_param("model", "Logistic Regression")
  mlflow.log_param("C", 0.5)
  mlflow.log_param("max_iter", 1000)
  mlflow.log_param("ngram_range", "(1,2)")

  # Log accuracy
  mlflow.log_metric("accuracy", accuracy_v2)

  # Log model
  mlflow.sklearn.log_model(
      model_v2,
      "model",
      registered_model_name="Spam_Message_Detector"
      )

  print("Model registered successfully!")

2026/09/21 06:02:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Model registered successfully!


Successfully registered model 'Spam_Message_Detector'.
Created version '1' of model 'Spam_Message_Detector'.


In [21]:
new_data = pd.DataFrame({
    "message":
     [
         "Your bank account will be blocked, verify now",
         "Congratulations! Claim your reward",
         "Your UPI account needs verification",
         "Free cashback available click now",
         "Please attend the meeting at 10 AM",
         "Can you send the assignment?",
         "Your KYC has expired click the link",
         "Let's meet after lunch",
         "Urgent! Your account will be suspended",
         "Please share today's notes"
         ],

    "label": [
        "spam",
        "spam",
        "spam",
        "spam",
        "ham",
        "ham",
        "spam",
        "ham",
        "spam",
        "ham"
        ]
    })

new_data

,message,label
0,"Your bank account will be blocked, verify now",spam
1,Congratulations! Claim your reward,spam
2,Your UPI account needs verification,spam
3,Free cashback available click now,spam
4,Please attend the meeting at 10 AM,ham
5,Can you send the assignment?,ham
6,Your KYC has expired click the link,spam
7,Let's meet after lunch,ham
8,Urgent! Your account will be suspended,spam
9,Please share today's notes,ham


In [22]:
new_predictions = model.predict(new_data["message"])
new_data["prediction"] = new_predictions
new_data

,message,label,prediction
0,"Your bank account will be blocked, verify now",spam,spam
1,Congratulations! Claim your reward,spam,spam
2,Your UPI account needs verification,spam,spam
3,Free cashback available click now,spam,spam
4,Please attend the meeting at 10 AM,ham,ham
5,Can you send the assignment?,ham,ham
6,Your KYC has expired click the link,spam,spam
7,Let's meet after lunch,ham,ham
8,Urgent! Your account will be suspended,spam,spam
9,Please share today's notes,ham,ham


In [23]:
new_accuracy = accuracy_score(
    new_data["label"],
    new_data["prediction"]
    )
print("New Data Accuracy:", new_accuracy)
print("\nClassification Report:")
print(
    classification_report
     (
         new_data["label"],
         new_data["prediction"]
     )
)

New Data Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00         4
        spam       1.00      1.00      1.00         6

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

